# Processing Tests

Experimentation with Pandas and PyTorch to read in CSV files and output chunked time series.

This code is not professional-grade, but is provided to the curious user as a loose reference with no warranties.

In [26]:
import csv
import pandas as pd
import torch

In [2]:
file_path = '/Users/ashtoncole/Downloads/GNN_READY/MAX_0051/combined_data.csv'
df = pd.read_csv(file_path)
df

,athlete_bool,track_id,mask,frame,wx,wy,vx,vy,sx,sy,bw,bh,conf,wx_m,wy_m,vx_m,vy_m,scale_factor
0,1,track_1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,track_1,0,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,track_1,0,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,track_1,0,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,track_1,0,8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
845,1,track_5,0,330,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
846,1,track_5,0,332,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
847,1,track_5,0,334,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
848,1,track_5,0,336,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
df['wx']

0      0.0
1      0.0
2      0.0
3      0.0
4      0.0
      ... 
845    0.0
846    0.0
847    0.0
848    0.0
849    0.0
Name: wx, Length: 850, dtype: float64

In [4]:
df.loc[0]

athlete_bool          1
track_id        track_1
mask                  0
frame                 0
wx                  0.0
wy                  0.0
vx                  0.0
vy                  0.0
sx                  0.0
sy                  0.0
bw                  0.0
bh                  0.0
conf                0.0
wx_m                0.0
wy_m                0.0
vx_m                0.0
vy_m                0.0
scale_factor        0.0
Name: 0, dtype: object

In [11]:
# Access single runner
runner_data = df[df['track_id'] == 'track_1']
runner_data

,athlete_bool,track_id,mask,frame,wx,wy,vx,vy,sx,sy,bw,bh,conf,wx_m,wy_m,vx_m,vy_m,scale_factor
0,1,track_1,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,1,track_1,0,2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,1,track_1,0,4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,1,track_1,0,6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,1,track_1,0,8,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,1,track_1,1,330,243.135284,1444.729736,-962.411743,173.952881,242.153137,1443.784546,206.839233,144.798218,0.720176,3.253703,19.333771,-12.879259,2.327885,0.013382
166,1,track_1,1,332,167.886093,1465.386475,-980.277832,191.476440,166.918396,1464.505615,252.470581,146.258667,0.701184,2.240708,19.557925,-13.083375,2.555559,0.013347
167,1,track_1,1,334,108.101967,1480.279053,-983.532288,198.411255,107.095879,1479.406860,214.191757,157.789551,0.773289,1.438937,19.703879,-13.091721,2.641037,0.013311
168,1,track_1,1,336,59.450249,1493.942017,-961.794312,206.763306,58.424061,1493.055298,116.848122,149.170776,0.701484,0.789217,19.832447,-12.768055,2.744834,0.013275


In [12]:
# num_frames
num_frames = len(runner_data)
num_frames

170

In [21]:
# num_particles
num_runners = int(len(df) / num_frames)
num_runners

5

In [22]:
# Check consistency
num_runners * num_frames

850

In [23]:
len(df)

850

In [27]:
# Create tensor
dim_state_reduced = 2 + 2 + 2
data = torch.zeros((num_frames, num_runners, dim_state_reduced), dtype=torch.float32)
data

tensor([[[0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.]],

        ...,

        [[0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0

In [24]:
# Looping across runners
for i in range(num_runners):
    runner_data = df[df['track_id'] == f'track_{i + 1}']
    print(runner_data)
    print('\n\n\n\n\n')

     athlete_bool track_id  mask  frame          wx           wy          vx  \
0               1  track_1     0      0    0.000000     0.000000    0.000000   
1               1  track_1     0      2    0.000000     0.000000    0.000000   
2               1  track_1     0      4    0.000000     0.000000    0.000000   
3               1  track_1     0      6    0.000000     0.000000    0.000000   
4               1  track_1     0      8    0.000000     0.000000    0.000000   
..            ...      ...   ...    ...         ...          ...         ...   
165             1  track_1     1    330  243.135284  1444.729736 -962.411743   
166             1  track_1     1    332  167.886093  1465.386475 -980.277832   
167             1  track_1     1    334  108.101967  1480.279053 -983.532288   
168             1  track_1     1    336   59.450249  1493.942017 -961.794312   
169             1  track_1     1    338   33.192745  1502.517578 -914.149780   

             vy          sx           s

In [32]:
# Access data for a single runner
i = 0 # First runner
df.loc[df['track_id'] == f'track_{i + 1}', ['wx', 'wy']] # x, y

,wx,wy
0,0.000000,0.000000
1,0.000000,0.000000
2,0.000000,0.000000
3,0.000000,0.000000
4,0.000000,0.000000
...,...,...
165,243.135284,1444.729736
166,167.886093,1465.386475
167,108.101967,1480.279053
168,59.450249,1493.942017


In [41]:
torch.from_numpy(df.loc[df['track_id'] == f'track_{i + 1}', ['bw', 'bh']].iloc[-5:].to_numpy())

tensor([[206.8392, 144.7982],
        [252.4706, 146.2587],
        [214.1918, 157.7896],
        [116.8481, 149.1708],
        [ 64.3795, 136.9877]], dtype=torch.float64)

In [44]:
res = torch.max(torch.from_numpy(df.loc[df['track_id'] == f'track_{i + 1}', ['bw', 'bh']].iloc[-5:].to_numpy()), dim=-1)
res

torch.return_types.max(
values=tensor([206.8392, 252.4706, 214.1918, 149.1708, 136.9877], dtype=torch.float64),
indices=tensor([0, 0, 0, 1, 1]))

In [45]:
res[0]

tensor([206.8392, 252.4706, 214.1918, 149.1708, 136.9877], dtype=torch.float64)

In [57]:
# Assign data to single runner
i = 0 # First runner
runner_data = df[df['track_id'] == f'track_{i + 1}']
data[:, i, 0:2] = torch.from_numpy(runner_data[['wx', 'wy']].to_numpy()) # x, y
data[:, i, 2:4] = torch.from_numpy(runner_data[['vx', 'vy']].to_numpy()) # vx, vy
data[:, i, 4] = torch.max(torch.from_numpy(runner_data[['bw', 'bh']].to_numpy()), dim=-1)[0] # max(bw, bh)
data[:, i, 5] = torch.from_numpy(runner_data['conf'].to_numpy()) # conf
data[:, i, :]

tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00],
        ...,
        [ 1.0810e+02,  1.4803e+03, -9.8353e+02,  1.9841e+02,  2.1419e+02,
          7.7329e-01],
        [ 5.9450e+01,  1.4939e+03, -9.6179e+02,  2.0676e+02,  1.4917e+02,
          7.0148e-01],
        [ 3.3193e+01,  1.5025e+03, -9.1415e+02,  2.0813e+02,  1.3699e+02,
          5.8931e-01]])

In [68]:
# Convert mask to bool
i = 0
df.loc[df['track_id'] == f'track_{i + 1}', 'mask'] == 1

0      False
1      False
2      False
3      False
4      False
       ...  
165     True
166     True
167     True
168     True
169     True
Name: mask, Length: 170, dtype: bool

In [75]:
# Combined mask
i = 0
mask = (df.loc[df['track_id'] == f'track_{i + 1}', 'mask'] == 1).to_numpy()
for i in range(num_runners):
    runner_mask = (df.loc[df['track_id'] == f'track_{i + 1}', 'mask'] == 1).to_numpy()
    print(f'runner {i + 1}:')
    print(runner_mask)
    mask = mask & runner_mask
mask

runner 1:
[False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  Tru

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,

In [94]:
# Chunking
# Create new data frame to hold which runners are visible at a given moment
masks = df.pivot(index='frame', columns='track_id', values='mask')
runner_columns = [f'track_{i + 1}' for i in range(num_runners)]
masks['current_state'] = masks[runner_columns].apply(tuple, axis=1)
masks

track_id,track_1,track_2,track_3,track_4,track_5,current_state
frame,,,,,,
0,0,0,0,0,0,"(0, 0, 0, 0, 0)"
2,0,0,0,0,0,"(0, 0, 0, 0, 0)"
4,0,0,0,0,0,"(0, 0, 0, 0, 0)"
6,0,0,0,0,0,"(0, 0, 0, 0, 0)"
8,0,0,0,0,0,"(0, 0, 0, 0, 0)"
...,...,...,...,...,...,...
330,1,0,0,0,0,"(1, 0, 0, 0, 0)"
332,1,0,0,0,0,"(1, 0, 0, 0, 0)"
334,1,0,0,0,0,"(1, 0, 0, 0, 0)"


In [112]:
# Compute chunks
masks['state_changed'] = masks['current_state'] != masks['current_state'].shift(1)
masks['chunk_id'] = masks['state_changed'].cumsum()
# Count number of runners
masks['num_runners'] = masks[runner_columns].sum(axis=1)
masks

track_id,track_1,track_2,track_3,track_4,track_5,current_state,state_changed,chunk_id,num_runners
frame,,,,,,,,,
0,0,0,0,0,0,"(0, 0, 0, 0, 0)",True,1,0
2,0,0,0,0,0,"(0, 0, 0, 0, 0)",False,1,0
4,0,0,0,0,0,"(0, 0, 0, 0, 0)",False,1,0
6,0,0,0,0,0,"(0, 0, 0, 0, 0)",False,1,0
8,0,0,0,0,0,"(0, 0, 0, 0, 0)",False,1,0
...,...,...,...,...,...,...,...,...,...
330,1,0,0,0,0,"(1, 0, 0, 0, 0)",False,14,1
332,1,0,0,0,0,"(1, 0, 0, 0, 0)",False,14,1
334,1,0,0,0,0,"(1, 0, 0, 0, 0)",False,14,1


In [149]:
min_chunk_length = 0
min_runners = 2
# Iterate through each chunk
for chunk_id, chunk_data in masks.groupby('chunk_id'):
    print(f'chunk id: {chunk_id} -----------------------')
    len_chunk = len(chunk_data)
    print(f'length of chunk: {len_chunk}')
    # Exclude if too short
    if len_chunk < min_chunk_length:
        print('Too short, excluding')
        continue
    runners_state = chunk_data[runner_columns].iloc[0] # Which runners are in
    print(f'runner state: {runners_state}')
    runners_present = [f'track_{j + 1}' for j in range(num_runners) if runners_state[f'track_{j + 1}'] == 1]
    print(f'present runners: {runners_present}')
    num_runners_chunk = chunk_data['num_runners'].iloc[0]
    print(f'number of runners: {num_runners_chunk}')
    # Exclude if no runners
    if num_runners_chunk < min_runners:
        print('Too short, excluding')
        continue
    frames_in_chunk = chunk_data.index
    print(f'associated video frames {frames_in_chunk}')
        
    # Create 3D tensor
    dim_state_reduced = 2 + 2 + 2 # (x, y, vx, vy, max(bw, bh), conf)
    states_chunk = torch.zeros((len_chunk, num_runners_chunk, dim_state_reduced), dtype=torch.float32)
    
    # Loop across runners to fill in data
    for i in range(num_runners_chunk):
        runner_data = df[(df['track_id'] == runners_present[i]) & df['frame'].isin(frames_in_chunk)]
        states_chunk[:, i, 0:2] = torch.from_numpy(runner_data[['wx', 'wy']].to_numpy()) # x, y
        states_chunk[:, i, 2:4] = torch.from_numpy(runner_data[['vx', 'vy']].to_numpy()) # vx, vy
        states_chunk[:, i, 4] = torch.max(torch.from_numpy(runner_data[['bw', 'bh']].to_numpy()), dim=-1)[0] # max(bw, bh)
        states_chunk[:, i, 5] = torch.from_numpy(runner_data['conf'].to_numpy()) # conf
    print(states_chunk)

chunk id: 1 -----------------------
length of chunk: 87
runner state: track_id
track_1    0
track_2    0
track_3    0
track_4    0
track_5    0
Name: 0, dtype: int64
present runners: []
number of runners: 0
Too short, excluding
chunk id: 2 -----------------------
length of chunk: 2
runner state: track_id
track_1    0
track_2    1
track_3    0
track_4    0
track_5    0
Name: 174, dtype: int64
present runners: ['track_2']
number of runners: 1
associated video frames Index([174, 176], dtype='int64', name='frame')
tensor([[[ 3.6490e+03,  1.2476e+03,  0.0000e+00,  0.0000e+00,  1.6582e+02,
           7.0854e-01]],

        [[ 3.5870e+03,  1.2602e+03, -9.3020e+02,  1.8865e+02,  2.0312e+02,
           8.0708e-01]]])
chunk id: 3 -----------------------
length of chunk: 1
runner state: track_id
track_1    0
track_2    0
track_3    0
track_4    0
track_5    0
Name: 178, dtype: int64
present runners: []
number of runners: 0
Too short, excluding
chunk id: 4 -----------------------
length of chunk: 